# Setup and Configs

In [1]:
import inspect
from feature_visualization import FeatureAnalyzer
from dotenv import load_dotenv
from pathlib import Path


/Users/adityaiyer/Desktop/Projects/sae-monosemantic/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -- Configs -- #
expansion_factor = 4
_lambda = 1e-4
gpt_dim = 768
feature_analysis_count = 25

project_root = Path().resolve().parent.parent
load_dotenv(project_root / ".env")

HF_dataset_path = f"thedarkknight7/SAE_monosemanticity_features_{expansion_factor}x_{_lambda}"
table_name=f"hf_{expansion_factor}x_{str(_lambda).replace('.', '_')}_full"

feature_analyzer = FeatureAnalyzer(
        HF_dataset_path = HF_dataset_path,
        db_name = str(project_root / "sae_feature_activations"),
        expansion_factor = expansion_factor
    )

feature_analyzer.create_features_table(table_name = table_name)
feature_analyzer.build_vocab_table()

/Users/adityaiyer/Desktop/Projects/sae-monosemantic/src/evaluation/feature_visualization.py:810: UserWarning: Table 'hf_4x_0_0001_full' already exists. Skipping creation.
  warnings.warn(f"Table '{table_name}' already exists. Skipping creation.", UserWarning)


In [3]:
methods_list = inspect.getmembers(feature_analyzer, predicate=inspect.ismethod)
for name, method in methods_list:
    print(name)

__init__
_table_exists
build_vocab_table
classify_activating_token
close
create_features_table
drop_column
drop_table
feature_similarity_correlation
feature_similarity_cosine_similarity
get_activation_distribution
get_activation_distribution_per_token_id
get_co_occuring_features
get_context_string
get_dead_features
get_feature_density
get_most_active_features
get_token_type_breakdown
get_top_activations
join_token_with_context
label_feature
llm_judge_classification
query
rank_features_by_selectivity
reconstruct_context_text
reconstruct_token_text


# Feature Analysis 

Here we filter table based on selectivity (mean activation / log(num activations + 1)), then take the top/bottom k features and analyze them using the following methods:
- get_top_activations(table_name, feature_id, top_k)	
- get_activation_distribution(table_name, feature_id)	
- get_co_occuring_features(table_name, feature_id)	
- get_token_type_breakdown(feature_id, table_name)
- label feature 	

In [4]:
min_activations = 10

feature_selective_ranked = feature_analyzer.rank_features_by_selectivity(
    table_name=table_name,
    min_activations=min_activations
)
top_k_df = feature_selective_ranked.head(feature_analysis_count)
bottom_k_df = feature_selective_ranked.tail(feature_analysis_count)

In [ ]:
def analyze_feature(feature_id: int) -> dict:
    top_act_df = feature_analyzer.get_top_activations(table_name, int(feature_id), top_k=25)
    top_act_df = feature_analyzer.reconstruct_token_text(top_act_df)
    top_act_df = feature_analyzer.reconstruct_context_text(top_act_df)

    labeled_df = feature_analyzer.label_feature(top_act_df, use_groq = False)
    llm_label = labeled_df["llm_label"].iloc[0]
    llm_reasoning = labeled_df["llm_reasoning"].iloc[0]

    stats = feature_analyzer.get_activation_distribution(table_name, int(feature_id))
    cooc = feature_analyzer.get_co_occuring_features(table_name, int(feature_id))
    breakdown = feature_analyzer.get_token_type_breakdown(int(feature_id), table_name)

    return {
        "feature_id": feature_id,
        "llm_label": llm_label,
        "llm_reasoning": llm_reasoning,
        "top_activations": labeled_df,
        "activation_stats": stats,
        "co_occurring_features": cooc,
        "token_type_breakdown": breakdown,
    }

### Top-k Features (Most Selective)

In [ ]:
top_feature_idx = 0  # Change this index and re-run to see the next feature

fid = top_k_df["feature_id"].iloc[top_feature_idx]
r = analyze_feature(fid)

print("=" * 80)
print(f"[{top_feature_idx + 1}/{len(top_k_df)}] Feature {r['feature_id']}: {r['llm_label']}")
print("=" * 80)

print(f"\n--- LLM Reasoning ---\n{r['llm_reasoning']}")

print("\n--- Top Activations ---")
display(r["top_activations"])

print("\n--- Activation Distribution Stats ---")
for k, v in r["activation_stats"].items():
    print(f"  {k}: {v}")

print("\n--- Co-occurring Features ---")
display(r["co_occurring_features"])

print("\n--- Token Type Breakdown ---")
for prop, counts in r["token_type_breakdown"].items():
    print(f"  {prop}: {counts}")

### Bottom-k Features (Least Selective)

In [ ]:
bottom_feature_idx = 0  # Change this index and re-run to see the next feature

fid = bottom_k_df["feature_id"].iloc[bottom_feature_idx]
r = analyze_feature(fid)

print("=" * 80)
print(f"[{bottom_feature_idx + 1}/{len(bottom_k_df)}] Feature {r['feature_id']}: {r['llm_label']}")
print("=" * 80)

print(f"\n--- LLM Reasoning ---\n{r['llm_reasoning']}")

print("\n--- Top Activations ---")
display(r["top_activations"])

print("\n--- Activation Distribution Stats ---")
for k, v in r["activation_stats"].items():
    print(f"  {k}: {v}")

print("\n--- Co-occurring Features ---")
display(r["co_occurring_features"])

print("\n--- Token Type Breakdown ---")
for prop, counts in r["token_type_breakdown"].items():
    print(f"  {prop}: {counts}")

# Dead Feature Proportions

In [ ]:
dead_df = feature_analyzer.get_dead_features()
dead_df

In [ ]:
dead_proportion = len(dead_df)/(expansion_factor*gpt_dim)
dead_proportion